# All-Site Low-Tide Exposure

For each site and each year from 2022 through 2025, keep Sentinel-2 scenes with catalogue cloud cover below 10%. Record the nearest predicted low tide within 3 hours and calculate dry area as SCL 4+5 pixels divided by the full rasterized site area. Save each site's CSV, RGB/SCL galleries, and dry-area scatter only in that site's folder.

In [1]:
import base64
import calendar
import html as html_module
import io
import os
import pathlib
import sys

import dotenv
import geopandas
import matplotlib.pyplot as plt
import numpy
import odc.stac
import pandas
import planetary_computer
import plotly.express as px
import rasterio.features
import requests
import xarray
from IPython.display import FileLink, display

working_path = pathlib.Path.cwd().resolve()
module_path = working_path / "scripts" if (working_path / "scripts").is_dir() else working_path.parent / "scripts"
if not module_path.is_dir():
    raise FileNotFoundError(f"Could not find the scripts directory from {working_path}")
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

import sentinel2
import utils

data_path = utils.get_data_path()
site_polygons_path = data_path / "site_polygons"
output_path = data_path / "analysis" / "all_sites_tide_exposure"
output_path.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv()
tide_api_key = os.environ.get("TIDE_API")
if not tide_api_key:
    raise ValueError("Set TIDE_API in the environment or project .env file before querying tides.")

print(f"Site polygons: {site_polygons_path}")
print(f"Output root: {output_path}")

Site polygons: C:\Local\repos\seagrass-detection\data\site_polygons
Output root: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure


## Survey Settings

The notebook processes the inclusive year range below. Every Sentinel-2 scene under the catalogue cloud cutoff is retained; a 3-hour line in each scatter marks the low-tide context window.

In [2]:
YEAR_START = 2022
YEAR_END = 2025
YEAR_RANGE = f"{YEAR_START}-{YEAR_END}"
MAX_ITEMS_PER_YEAR = 400
MAX_CLOUD_COVER_PCT = 10.0
CANDIDATE_WINDOW_HRS = 4.0
TIDE_SEARCH_RADIUS_HRS = 12
LOCAL_TIME_ZONE = "Pacific/Auckland"
OUTPUT_REVISION = "lowtide-duration-v4"

site_polygon_files = sorted(site_polygons_path.glob("*_polygon.gpkg"))
if not site_polygon_files:
    raise FileNotFoundError(f"No site polygons found under {site_polygon_files}")
site_names = [path.stem.removesuffix("_polygon") for path in site_polygon_files]

print(f"Year range: {YEAR_RANGE}; sites: {len(site_polygon_files)}")
print(f"STAC max_items per year/site: {MAX_ITEMS_PER_YEAR}")
print(f"Catalogue cloud cutoff: <{MAX_CLOUD_COVER_PCT:g}%")
print(f"Low-tide reference range: +/-{CANDIDATE_WINDOW_HRS:g} hours")

Year range: 2022-2025; sites: 20
STAC max_items per year/site: 400
Catalogue cloud cutoff: <10%
Low-tide reference range: +/-3 hours


## Tide Event Helpers

Fetch NIWA high/low events in requests of at most 31 days, cache by site and month, and calculate the nearest low event, signed offset, and formatted duration.

In [3]:
def fetch_tide_values(start_date, number_of_days, latitude, longitude):
    if not 1 <= number_of_days <= 31:
        raise ValueError("NIWA tide requests must cover between 1 and 31 days.")
    response = requests.get(
        sentinel2.TIDE_API_STUB,
        params={
            "lat": latitude,
            "long": longitude,
            "datum": "MSL",
            "numberOfDays": number_of_days,
            "apikey": tide_api_key,
            "startDate": start_date.strftime("%Y-%m-%d"),
        },
        timeout=30,
    )
    response.raise_for_status()
    tide_data = pandas.DataFrame.from_records(response.json().get("values", []))
    if tide_data.empty:
        return tide_data
    tide_data["time"] = pandas.to_datetime(tide_data["time"], utc=True)
    tide_data["value"] = pandas.to_numeric(tide_data["value"], errors="raise")
    return tide_data


def tide_predictions_for_month(timestamp_utc, latitude, longitude, tide_cache):
    timestamp_utc = pandas.Timestamp(timestamp_utc)
    timestamp_utc = (
        timestamp_utc.tz_localize("UTC")
        if timestamp_utc.tzinfo is None
        else timestamp_utc.tz_convert("UTC")
    )
    month_start = timestamp_utc.normalize().replace(day=1)
    request_start = month_start - pandas.Timedelta(days=1)
    requested_days = calendar.monthrange(month_start.year, month_start.month)[1] + 2
    cache_key = (latitude, longitude, month_start.strftime("%Y-%m"))
    if cache_key not in tide_cache:
        first_days = min(requested_days, 31)
        tide_parts = [fetch_tide_values(request_start, first_days, latitude, longitude)]
        if requested_days > first_days:
            overlap_days = 1
            next_start = request_start + pandas.Timedelta(days=first_days - overlap_days)
            next_days = requested_days - first_days + overlap_days
            tide_parts.append(fetch_tide_values(next_start, next_days, latitude, longitude))
        tide_data = pandas.concat(tide_parts, ignore_index=True)
        if tide_data.empty:
            raise ValueError(f"NIWA returned no tide events for {month_start:%Y-%m}.")
        tide_cache[cache_key] = (
            tide_data.sort_values("time").drop_duplicates("time", keep="last").reset_index(drop=True)
        )
    return tide_cache[cache_key]


def nearest_low_tide(timestamp_utc, tide_data):
    timestamp_utc = pandas.Timestamp(timestamp_utc)
    timestamp_utc = (
        timestamp_utc.tz_localize("UTC")
        if timestamp_utc.tzinfo is None
        else timestamp_utc.tz_convert("UTC")
    )
    search_window = tide_data[
        tide_data["time"].between(
            timestamp_utc - pandas.Timedelta(hours=TIDE_SEARCH_RADIUS_HRS),
            timestamp_utc + pandas.Timedelta(hours=TIDE_SEARCH_RADIUS_HRS),
        )
    ].copy()
    if search_window.empty:
        raise ValueError(f"No NIWA tide events near {timestamp_utc}.")
    is_local_minimum = (tide_data["value"] <= tide_data["value"].shift(1)) & (
        tide_data["value"] <= tide_data["value"].shift(-1)
    )
    low_points = tide_data.loc[
        is_local_minimum & tide_data["time"].isin(search_window["time"])
    ]
    if low_points.empty:
        low_event = search_window.nsmallest(1, "value").iloc[0]
    else:
        low_index = (low_points["time"] - timestamp_utc).abs().idxmin()
        low_event = low_points.loc[low_index]
    hours_from_low = (timestamp_utc - low_event["time"]).total_seconds() / 3600
    total_minutes = int(round(abs(hours_from_low) * 60))
    hours, minutes = divmod(total_minutes, 60)
    direction = "at" if total_minutes == 0 else "before" if hours_from_low < 0 else "after"
    nearest_event = tide_data.loc[(tide_data["time"] - timestamp_utc).abs().idxmin()]
    return {
        "tide_height_msl": float(nearest_event["value"]),
        "nearest_low_tide_time_utc": low_event["time"],
        "nearest_low_tide_height_msl": float(low_event["value"]),
        "hours_from_low_tide": hours_from_low,
        "duration_from_low_tide": f"{hours}h {minutes:02d}m {direction}",
    }

## Process Each Site

All scenes meeting the cloud-cover cutoff are measured; low tide is recorded as context and does not exclude scenes.

In [4]:
def analyze_site(site_polygon_file):
    site_name = site_polygon_file.stem.removesuffix("_polygon")
    site_output_dir = output_path / site_name
    site_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{site_name}: search {YEAR_RANGE}")

    site_polygon = geopandas.read_file(site_polygon_file)
    if site_polygon.crs is None or site_polygon.empty:
        raise ValueError(f"Site polygon must be non-empty and have a CRS: {site_polygon_file}")
    polygon_wgs84 = site_polygon.to_crs(utils.CRS_WSG)
    centroid = polygon_wgs84.geometry.union_all().centroid
    latitude, longitude = float(centroid.y), float(centroid.x)
    search_geometry = site_polygon.buffer(sentinel2.S2_RESOLUTION).to_crs(utils.CRS_WSG)

    items_by_id = {}
    cloud_by_date = {}
    for year in range(YEAR_START, YEAR_END + 1):
        search_results = sentinel2.stac_search_with_retry(
            url=sentinel2.CATALOGUE_URL,
            max_items=MAX_ITEMS_PER_YEAR,
            collections=[sentinel2.COLLECTION],
            bbox=search_geometry.total_bounds,
            datetime=f"{year}-01-01/{year}-12-31",
            sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
            get_collection=True,
        )
        for item in search_results.items:
            raw_cloud_cover = item.properties.get("eo:cloud_cover")
            if raw_cloud_cover is None:
                continue
            cloud_cover = float(raw_cloud_cover)
            if cloud_cover >= MAX_CLOUD_COVER_PCT:
                continue
            acquisition_time = pandas.to_datetime(item.properties["datetime"], utc=True)
            scene_date = acquisition_time.strftime("%Y-%m-%d")
            items_by_id[item.id] = item
            current_cloud_cover = cloud_by_date.get(scene_date)
            if current_cloud_cover is None or cloud_cover < current_cloud_cover:
                cloud_by_date[scene_date] = cloud_cover

    print(f"  Distinct cloud-filtered dates: {len(cloud_by_date)}")
    if not items_by_id:
        print(f"  No scenes below {MAX_CLOUD_COVER_PCT:g}% cloud cover")
        return pandas.DataFrame()

    satellite_data = odc.stac.load(
        list(items_by_id.values()),
        bbox=search_geometry.total_bounds,
        bands=sentinel2.RGB_BANDS + ["SCL"],
        chunks={},
        groupby="solar_day",
        resolution=sentinel2.S2_RESOLUTION,
        dtype="uint16",
        nodata=0,
        patch_url=planetary_computer.sign,
    )
    satellite_data = satellite_data.rio.clip(
        site_polygon.to_crs(satellite_data.rio.crs).geometry,
        all_touched=True,
        drop=True,
    ).load()
    if satellite_data.sizes.get("time", 0) == 0:
        print("  No selected scenes intersect the site polygon")
        return pandas.DataFrame()

    site_polygon_projected = site_polygon.to_crs(satellite_data.rio.crs)
    site_mask_values = rasterio.features.geometry_mask(
        [geometry.__geo_interface__ for geometry in site_polygon_projected.geometry],
        out_shape=(satellite_data.sizes["y"], satellite_data.sizes["x"]),
        transform=satellite_data.rio.transform(),
        invert=True,
        all_touched=True,
    )
    site_mask = xarray.DataArray(
        site_mask_values,
        coords={"y": satellite_data["y"], "x": satellite_data["x"]},
        dims=("y", "x"),
    )
    site_pixel_count = int(site_mask.sum())
    if site_pixel_count == 0:
        raise ValueError(f"No raster pixels fall inside {site_polygon_file}")

    scl = satellite_data["SCL"]
    dry_pixels = (scl.isin([4, 5]) & site_mask).sum(dim=["x", "y"]).compute()
    scl_coverage = ((scl > 0) & site_mask).sum(dim=["x", "y"]).compute()
    scene_dates = satellite_data.time.dt.strftime("%Y-%m-%d").values.tolist()
    tide_cache = {}
    records = []
    for scene_index, scene_date in enumerate(scene_dates):
        timestamp = pandas.Timestamp(satellite_data.time.isel(time=scene_index).values)
        timestamp = timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")
        tide_data = tide_predictions_for_month(timestamp, latitude, longitude, tide_cache)
        low_tide = nearest_low_tide(timestamp, tide_data)
        n_dry = int(dry_pixels.isel(time=scene_index))
        n_covered = int(scl_coverage.isel(time=scene_index))
        records.append({
            "site": site_name,
            "date": scene_date,
            "year": int(scene_date[:4]),
            "satellite_time_utc": timestamp,
            "cloud_cover_pct": cloud_by_date.get(scene_date, numpy.nan),
            "nearest_low_tide_time_utc": low_tide["nearest_low_tide_time_utc"],
            "nearest_low_tide_height_msl": low_tide["nearest_low_tide_height_msl"],
            "hours_from_low_tide": low_tide["hours_from_low_tide"],
            "duration_from_low_tide": low_tide["duration_from_low_tide"],
            "dry_pixels_scl_4_5": n_dry,
            "dry_area_pct": 100 * n_dry / site_pixel_count,
            "scl_coverage_pct": 100 * n_covered / site_pixel_count,
            "site_pixels": site_pixel_count,
        })

    site_data = pandas.DataFrame.from_records(records).sort_values("satellite_time_utc").reset_index(drop=True)
    site_csv = site_output_dir / f"{site_name}_{YEAR_RANGE}_dry_area_{OUTPUT_REVISION}.csv"
    site_data.to_csv(site_csv, index=False)
    print(f"  Saved {len(site_data)} scene rows: {site_csv}")

    scl_classes = [
        (0, "No data", "#000000"),
        (1, "Saturated/defective", "#ff0000"),
        (2, "Dark area", "#2f2f2f"),
        (3, "Cloud shadow", "#643200"),
        (4, "Vegetation", "#00a000"),
        (5, "Not vegetated", "#ffe65a"),
        (6, "Water", "#0000ff"),
        (7, "Unclassified", "#808080"),
        (8, "Cloud, medium probability", "#c0c0c0"),
        (9, "Cloud, high probability", "#ffffff"),
        (10, "Thin cirrus", "#64c8ff"),
        (11, "Snow/ice", "#ff96ff"),
    ]
    scl_legend = "".join(
        f'<span class="legend-item"><i style="background:{color}"></i>{code}: '
        f'{html_module.escape(label)}</span>'
        for code, label, color in scl_classes
    )
    rgb_cards = []
    scl_cards = []
    for scene_index, row in site_data.iterrows():
        scene_date = html_module.escape(str(row["date"]))
        low_local = pandas.Timestamp(row["nearest_low_tide_time_utc"]).tz_convert(LOCAL_TIME_ZONE)
        caption = (
            f"Dry Area: {row['dry_area_pct']:.1f}%<br>"
            f"Low {low_local:%d %b %H:%M} | {row['duration_from_low_tide']} | "
            f"{row['nearest_low_tide_height_msl']:.2f} m MSL"
        )
        rgb = numpy.stack(
            [satellite_data[band].isel(time=scene_index).values for band in sentinel2.RGB_BANDS],
            axis=-1,
        ).astype(numpy.float32)
        rgb_valid = numpy.isfinite(rgb).all(axis=-1) & (rgb > 0).all(axis=-1)
        rgb_display = numpy.ones_like(rgb)
        if rgb_valid.any():
            for channel in range(3):
                pixels = rgb[..., channel][rgb_valid]
                low, high = numpy.percentile(pixels, [2, 98])
                if high > low:
                    rgb_display[..., channel] = numpy.clip(
                        (rgb[..., channel] - low) / (high - low), 0, 1
                    )
        rgb_display[~rgb_valid] = 1
        rgb_buffer = io.BytesIO()
        rgb_figure, rgb_axis = plt.subplots(figsize=(5, 3.75))
        rgb_axis.imshow(rgb_display)
        rgb_axis.axis("off")
        rgb_figure.savefig(
            rgb_buffer, format="jpg", dpi=90, bbox_inches="tight", pad_inches=0,
            pil_kwargs={"quality": 84, "optimize": True},
        )
        plt.close(rgb_figure)
        rgb_encoded = base64.b64encode(rgb_buffer.getvalue()).decode("ascii")
        rgb_cards.append(
            f'<article class="scene"><h2>{scene_date}</h2>'
            f'<img src="data:image/jpeg;base64,{rgb_encoded}" alt="{site_name} RGB {scene_date}">' 
            f'<div class="caption">{caption}</div></article>'
        )

        scl_array = satellite_data["SCL"].isel(time=scene_index).values
        scl_rgb = numpy.full((*scl_array.shape, 3), 238, dtype=numpy.uint8)
        for class_code, _, color in scl_classes:
            color_rgb = tuple(int(color[index:index + 2], 16) for index in (1, 3, 5))
            scl_rgb[(scl_array == class_code) & site_mask_values] = color_rgb
        scl_buffer = io.BytesIO()
        scl_figure, scl_axis = plt.subplots(figsize=(5, 3.75))
        scl_axis.imshow(scl_rgb)
        scl_axis.axis("off")
        scl_figure.savefig(
            scl_buffer, format="jpg", dpi=90, bbox_inches="tight", pad_inches=0,
            pil_kwargs={"quality": 90, "optimize": True},
        )
        plt.close(scl_figure)
        scl_encoded = base64.b64encode(scl_buffer.getvalue()).decode("ascii")
        scl_cards.append(
            f'<article class="scene"><h2>{scene_date}</h2>'
            f'<img src="data:image/jpeg;base64,{scl_encoded}" alt="{site_name} SCL {scene_date}">' 
            f'<div class="caption">{caption}</div></article>'
        )

    gallery_css = """
:root { color-scheme: light; font-family: "Segoe UI", sans-serif; color: #202a33; background: #f3f5f6; }
body { margin: 0; padding: 20px; }
header, .gallery, .legend { max-width: 1500px; margin: 0 auto 16px; }
h1 { margin: 0 0 6px; font-size: 22px; }
p { margin: 0; color: #56616b; }
.gallery { display: grid; grid-template-columns: repeat(auto-fill, minmax(min(320px, 100%), 1fr)); gap: 14px; }
.scene { min-width: 0; overflow: hidden; background: white; border: 1px solid #d5dce1; border-radius: 5px; }
.scene h2 { margin: 10px 12px 0; font-size: 15px; }
.scene img { display: block; width: 100%; aspect-ratio: 4 / 3; object-fit: contain; background: #f7f8f8; }
.caption { padding: 8px 12px 11px; font-size: 12px; line-height: 1.4; }
.legend { display: flex; flex-wrap: wrap; gap: 6px 12px; padding: 10px; background: white; border: 1px solid #d5dce1; }
.legend-item { display: inline-flex; align-items: center; gap: 5px; font-size: 11px; }
.legend-item i { display: inline-block; width: 14px; height: 14px; border: 1px solid #7b858c; }
@media (max-width: 600px) { body { padding: 10px; } }
"""

    def write_gallery(gallery_title, cards, output_file, legend_html=""):
        page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{html_module.escape(site_name)} {YEAR_RANGE} {html_module.escape(gallery_title)}</title>
<style>{gallery_css}</style></head><body>
<header><h1>{html_module.escape(site_name)}: {YEAR_RANGE} {html_module.escape(gallery_title)}</h1>
<p>{len(cards)} scenes; catalogue cloud cover under {MAX_CLOUD_COVER_PCT:g}%.</p></header>
{legend_html}<main class="gallery">{''.join(cards)}</main></body></html>"""
        output_file.write_text(page, encoding="utf-8")
        print(f"  Saved gallery: {output_file}")

    rgb_gallery_file = site_output_dir / f"{site_name}_{YEAR_RANGE}_rgb_{OUTPUT_REVISION}.html"
    scl_gallery_file = site_output_dir / f"{site_name}_{YEAR_RANGE}_scl_{OUTPUT_REVISION}.html"
    write_gallery("RGB", rgb_cards, rgb_gallery_file)
    write_gallery("SCL", scl_cards, scl_gallery_file, f'<div class="legend">{scl_legend}</div>')

    scatter_data = site_data.copy()
    scatter_data["abs_hours_from_low_tide"] = scatter_data["hours_from_low_tide"].abs()
    scatter_data["date_color"] = pandas.to_datetime(
        scatter_data["satellite_time_utc"], utc=True
    ).astype("int64") / 1e9
    color_ticks = numpy.linspace(scatter_data["date_color"].min(), scatter_data["date_color"].max(), 5)
    color_tick_labels = pandas.to_datetime(color_ticks, unit="s", utc=True).strftime("%d %b %Y").tolist()
    scatter = px.scatter(
        scatter_data,
        x="abs_hours_from_low_tide",
        y="dry_area_pct",
        color="date_color",
        color_continuous_scale="Viridis",
        hover_name="date",
        custom_data=["duration_from_low_tide"],
        labels={
            "abs_hours_from_low_tide": "Absolute hours from low tide",
            "dry_area_pct": "Dry area (% of full site)",
            "date_color": "Acquisition date",
        },
        title=f"{site_name}: Dry Area vs Low-Tide Duration ({YEAR_RANGE})",
    )
    scatter.update_traces(
        marker={"size": 10, "opacity": 0.85, "line": {"width": 0.6, "color": "white"}},
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "Dry Area: %{y:.1f}%<br>"
            "Duration from low tide: %{customdata[0]}<extra></extra>"
        ),
    )
    scatter.update_layout(
        height=650,
        hovermode="closest",
        coloraxis_colorbar={
            "title": "Acquisition date",
            "tickvals": color_ticks.tolist(),
            "ticktext": color_tick_labels,
        },
        yaxis={"range": [0, 100]},
    )
    scatter.add_vline(
        x=CANDIDATE_WINDOW_HRS,
        line_dash="dot",
        line_color="#b23a48",
        annotation_text="3-hour range",
        annotation_position="top left",
    )
    scatter_file = site_output_dir / f"{site_name}_{YEAR_RANGE}_dry_vs_low_tide_{OUTPUT_REVISION}.html"
    scatter.write_html(scatter_file, include_plotlyjs=True, auto_open=False)
    print(f"  Saved scatter: {scatter_file}")
    return site_data

## Process All Sites

Sites run sequentially. Per-site CSVs and plots are stored in each site's folder; the aggregate CSV is stored at the analysis root.

In [5]:
site_results = {}
site_errors = []
for site_polygon_file in site_polygon_files:
    site_name = site_polygon_file.stem.removesuffix("_polygon")
    try:
        site_data = analyze_site(site_polygon_file)
        if site_data.empty:
            print(f"{site_name}: no scenes met the cloud cutoff")
            continue
        site_results[site_name] = site_data
        display(site_data.sort_values("dry_area_pct", ascending=False))
    except Exception as error:
        site_errors.append({"site": site_name, "error": str(error)})
        print(f"  ERROR for {site_name}: {error}")

print(f"\nCompleted {len(site_results)} of {len(site_polygon_files)} sites.")
if site_errors:
    print("Sites with errors:")
    display(pandas.DataFrame.from_records(site_errors))

print("Per-site outputs:")
for site_name in site_results:
    site_output_dir = output_path / site_name
    for output_file in sorted(site_output_dir.iterdir()):
        display(FileLink(str(output_file)))


CatlinsLake: search 2022-2025
  Distinct cloud-filtered dates: 45
  Saved 45 scene rows: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsLake\CatlinsLake_2022-2025_dry_area_lowtide-duration-v4.csv
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsLake\CatlinsLake_2022-2025_rgb_lowtide-duration-v4.html
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsLake\CatlinsLake_2022-2025_scl_lowtide-duration-v4.html
  Saved scatter: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsLake\CatlinsLake_2022-2025_dry_vs_low_tide_lowtide-duration-v4.html


,site,date,year,satellite_time_utc,cloud_cover_pct,nearest_low_tide_time_utc,nearest_low_tide_height_msl,hours_from_low_tide,duration_from_low_tide,dry_pixels_scl_4_5,dry_area_pct,scl_coverage_pct,site_pixels
25,CatlinsLake,2023-08-29,2023,2023-08-29 22:37:19.024000+00:00,0.054977,2023-08-29 19:27:00+00:00,-1.08,3.171951,3h 10m after,6737,15.197040,100.000000,44331
35,CatlinsLake,2024-08-30,2024,2024-08-30 22:25:49.024000+00:00,1.864714,2024-08-30 18:31:00+00:00,-0.76,3.913618,3h 55m after,6526,14.721076,100.000000,44331
36,CatlinsLake,2024-09-29,2024,2024-09-29 22:25:49.024000+00:00,0.005375,2024-09-29 18:51:00+00:00,-0.76,3.580284,3h 35m after,5235,11.808892,100.000000,44331
26,CatlinsLake,2023-09-08,2023,2023-09-08 22:37:09.024000+00:00,2.688654,2023-09-08 16:15:00+00:00,-0.63,6.369173,6h 22m after,5151,11.619409,100.000000,44331
40,CatlinsLake,2025-04-10,2025,2025-04-10 22:37:09.024000+00:00,4.059831,2025-04-10 19:39:00+00:00,-0.83,2.969173,2h 58m after,4510,10.173468,100.000000,44331
42,CatlinsLake,2025-09-07,2025,2025-09-07 22:37:09.024000+00:00,2.634845,2025-09-07 20:34:00+00:00,-1.01,2.052507,2h 03m after,4400,9.925334,100.000000,44331
20,CatlinsLake,2023-05-03,2023,2023-05-03 22:25:41.024000+00:00,2.077316,2023-05-03 19:30:00+00:00,-0.81,2.928062,2h 56m after,4392,9.907288,100.000000,44331
19,CatlinsLake,2023-03-07,2023,2023-03-07 22:37:01.024000+00:00,0.314562,2023-03-07 21:24:00+00:00,-0.81,1.216951,1h 13m after,4376,9.871196,100.000000,44331
9,CatlinsLake,2022-09-10,2022,2022-09-10 22:25:49.024000+00:00,3.049057,2022-09-10 20:57:00+00:00,-1.09,1.480284,1h 29m after,4337,9.783222,100.000000,44331
21,CatlinsLake,2023-05-06,2023,2023-05-06 22:37:11.024000+00:00,0.689979,2023-05-06 21:32:00+00:00,-0.88,1.086396,1h 05m after,4337,9.783222,99.990977,44331



CatlinsRiverMouth: search 2022-2025
  Distinct cloud-filtered dates: 66
  Saved 66 scene rows: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsRiverMouth\CatlinsRiverMouth_2022-2025_dry_area_lowtide-duration-v4.csv
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsRiverMouth\CatlinsRiverMouth_2022-2025_rgb_lowtide-duration-v4.html
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsRiverMouth\CatlinsRiverMouth_2022-2025_scl_lowtide-duration-v4.html
  Saved scatter: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\CatlinsRiverMouth\CatlinsRiverMouth_2022-2025_dry_vs_low_tide_lowtide-duration-v4.html


,site,date,year,satellite_time_utc,cloud_cover_pct,nearest_low_tide_time_utc,nearest_low_tide_height_msl,hours_from_low_tide,duration_from_low_tide,dry_pixels_scl_4_5,dry_area_pct,scl_coverage_pct,site_pixels
31,CatlinsRiverMouth,2023-06-15,2023,2023-06-15 22:37:11.024000+00:00,9.683786,2023-06-15 19:04:00+00:00,-0.85,3.553062,3h 33m after,2069,33.093410,100.0,6252
55,CatlinsRiverMouth,2025-04-10,2025,2025-04-10 22:37:09.024000+00:00,0.002813,2025-04-10 19:41:00+00:00,-0.82,2.935840,2h 56m after,2016,32.245681,100.0,6252
63,CatlinsRiverMouth,2025-09-07,2025,2025-09-07 22:37:09.024000+00:00,2.634845,2025-09-07 20:36:00+00:00,-1.00,2.019173,2h 01m after,1865,29.830454,100.0,6252
62,CatlinsRiverMouth,2025-08-25,2025,2025-08-25 22:25:49.024000+00:00,6.799818,2025-08-25 22:35:00+00:00,-0.83,-0.153049,0h 09m before,1859,29.734485,100.0,6252
64,CatlinsRiverMouth,2025-10-09,2025,2025-10-09 22:26:01.025000+00:00,0.998645,2025-10-09 22:50:00+00:00,-1.01,-0.399715,0h 24m before,1821,29.126679,100.0,6252
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,CatlinsRiverMouth,2022-05-01,2022,2022-05-01 22:37:11.024000+00:00,8.831602,2022-05-01 21:37:00+00:00,-0.84,1.003062,1h 00m after,740,11.836212,100.0,6252
39,CatlinsRiverMouth,2023-11-09,2023,2023-11-09 22:25:41.024000+00:00,8.020958,2023-11-09 18:19:00+00:00,-0.65,4.111396,4h 07m after,202,3.230966,100.0,6252
42,CatlinsRiverMouth,2024-01-26,2024,2024-01-26 22:37:09.024000+00:00,2.877543,2024-01-26 21:59:00+00:00,-0.80,0.635840,0h 38m after,16,0.255918,100.0,6252
2,CatlinsRiverMouth,2022-03-19,2022,2022-03-19 22:25:51.024000+00:00,4.613432,2022-03-19 22:05:00+00:00,-0.93,0.347507,0h 21m after,0,0.000000,100.0,6252



Childrens: search 2022-2025
  Distinct cloud-filtered dates: 89
  Saved 89 scene rows: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\Childrens\Childrens_2022-2025_dry_area_lowtide-duration-v4.csv
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\Childrens\Childrens_2022-2025_rgb_lowtide-duration-v4.html
  Saved gallery: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\Childrens\Childrens_2022-2025_scl_lowtide-duration-v4.html
  Saved scatter: C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\Childrens\Childrens_2022-2025_dry_vs_low_tide_lowtide-duration-v4.html


...

C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\ThePoint_Feb26\ThePoint_Feb26_2022-2025_scl_lowtide-duration-v4.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\ThePoint_Nov25\ThePoint_Nov25_2022-2025_dry_area_lowtide-duration-v4.csv

C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\ThePoint_Nov25\ThePoint_Nov25_2022-2025_dry_vs_low_tide_lowtide-duration-v4.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\ThePoint_Nov25\ThePoint_Nov25_2022-2025_rgb_lowtide-duration-v4.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites_tide_exposure\ThePoint_Nov25\ThePoint_Nov25_2022-2025_scl_lowtide-duration-v4.html